# 1. Import Required Libraries
Import pandas, numpy, os, and scikit-learn preprocessing modules for data manipulation and feature engineering.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# 2. Load the Merged Data
Read the merged CSV data file using pandas for further processing.

In [2]:
input_path = os.path.join("..", "integration", "merged_data.csv")
df = pd.read_csv(input_path)
df.head()

C:\Users\drket\AppData\Local\Temp\ipykernel_3260\1406275813.py:2: DtypeWarning: Columns (1,2,4,10,16,17,18,19,20,21,22,39,53,63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


,ID,State,City,location,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Year_y,Month_y,Day,Night,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess
0,NaN,NaN,NaN,"Madhurangan Apartment ,Ambegaon, Pune",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,"Manganahalli Sriram Layout ,Ullal Uppana...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,"sona Building,Bhayandar West, Mumbai",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,"Sec 2 Pooja apartment Bhosari ,Indrayani Nag...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,"A N SWAGATH,Gubbalala, Subramanyapura,Bangalore",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 3. Drop Irrelevant Columns
Remove columns that are not useful for modeling, such as IDs, timestamps, and location coordinates.

In [3]:
irrelevant_cols = [
    "id", "timestamp", "date", "name", "address", "location", "Unnamed: 0", "serial_no", "lat", "lon", "latitude", "longitude"
]
irrelevant_cols = [col for col in irrelevant_cols if col in df.columns]
df = df.drop(columns=irrelevant_cols)
df.head()

,ID,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,...,Year_y,Month_y,Day,Night,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 4. Add New Features
Create new features such as price per square foot, bedrooms per square foot, polynomial features, total rooms, property age, binned property age, and inflation-adjusted price.

In [4]:
# Price per sqft, bedrooms per sqft, polynomial features
if "area" in df.columns and "bedrooms" in df.columns:
    df["price_per_sqft"] = df["price"] / df["area"]
    df["bedrooms_per_sqft"] = df["bedrooms"] / df["area"]
    df["area_sq"] = df["area"] ** 2
    df["bedrooms_sq"] = df["bedrooms"] ** 2

# Total rooms
if "bedrooms" in df.columns and "bathrooms" in df.columns and "total_rooms" not in df.columns:
    df["total_rooms"] = df["bedrooms"] + df["bathrooms"]

# Property age, binned age, inflation-adjusted price
if "year_built" in df.columns:
    current_year = pd.Timestamp.now().year
    df["property_age"] = current_year - df["year_built"]
    df["property_age_bin"] = pd.cut(df["property_age"], bins=[0, 5, 15, 30, 100], labels=["new", "mid", "old", "very_old"])
    inflation_index = {year: 1 + 0.05 * (current_year - year) for year in df["year_built"].unique()}
    df["inflation_adjusted_price"] = df.apply(lambda x: x["price"] * inflation_index.get(x["year_built"], 1), axis=1)
df.head()

,ID,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,...,Year_y,Month_y,Day,Night,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 5. Environmental Features
Add or impute environmental features like air quality, noise level, crime rate, and water quality.

In [5]:
for col in ["air_quality", "noise_level", "crime_rate", "water_quality"]:
    if col not in df.columns:
        df[col] = np.nan
df.head()

,ID,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,...,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess,air_quality,noise_level,crime_rate,water_quality
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 6. Interaction Features
Create interaction features between area and air quality, and price per square foot and crime rate.

In [6]:
if "area" in df.columns and "air_quality" in df.columns:
    df["area_x_air_quality"] = df["area"] * df["air_quality"]
if "price_per_sqft" in df.columns and "crime_rate" in df.columns:
    df["price_per_sqft_x_crime_rate"] = df["price_per_sqft"] * df["crime_rate"]
df.head()

,ID,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,...,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess,air_quality,noise_level,crime_rate,water_quality
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 7. Log Transforms for Skewed Features
Apply log transformations to skewed features such as area, price, and price per square foot.

In [7]:
for col in ["area", "price", "price_per_sqft"]:
    if col in df.columns:
        df[f"log_{col}"] = np.log1p(df[col])
df.head()

,ID,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,...,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess,air_quality,noise_level,crime_rate,water_quality
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 8. Boolean Flags
Create a boolean flag for luxury properties based on price per square foot quantile.

In [8]:
if "price_per_sqft" in df.columns:
    df["is_luxury"] = (df["price_per_sqft"] > df["price_per_sqft"].quantile(0.9)).astype(int)
df.head()

,ID,State,City,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,...,DayLimit,NightLimit,Date_y,StationEncoded,DayExcess,NightExcess,air_quality,noise_level,crime_rate,water_quality
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 9. Categorical Encoding
Encode categorical features using one-hot encoding for low cardinality and label encoding for high cardinality.

In [9]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "property_age_bin"]
for col in categorical_cols:
    if df[col].nunique() < 10:
        dummies = pd.get_dummies(df[col], prefix=col)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(columns=[col])
    else:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
df.head()

,ID,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,...,Owner_Type_Owner,Availability_Status_Ready_to_Move,Availability_Status_Under_Construction,Balcony_No,Balcony_Yes,AQI_Bucket_Good,AQI_Bucket_Moderate,AQI_Bucket_Poor,AQI_Bucket_Satisfactory,AQI_Bucket_Very Poor
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,False
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,False


# 10. Statistical Aggregations
Compute mean and standard deviation of price per location if location data is available.

In [10]:
if "location" in df.columns and "price" in df.columns:
    df["mean_price_location"] = df.groupby("location")["price"].transform("mean")
    df["std_price_location"] = df.groupby("location")["price"].transform("std")
df.head()

,ID,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,...,Owner_Type_Owner,Availability_Status_Ready_to_Move,Availability_Status_Under_Construction,Balcony_No,Balcony_Yes,AQI_Bucket_Good,AQI_Bucket_Moderate,AQI_Bucket_Poor,AQI_Bucket_Satisfactory,AQI_Bucket_Very Poor
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,False
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,True,False,False,False,False,False
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,False,False,False


# 11. Fill Missing Values
Fill missing values in the dataset using the median strategy for numerical columns.

In [11]:
df = df.fillna(df.median(numeric_only=True))
df.head()

,ID,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,...,Owner_Type_Owner,Availability_Status_Ready_to_Move,Availability_Status_Under_Construction,Balcony_No,Balcony_Yes,AQI_Bucket_Good,AQI_Bucket_Moderate,AQI_Bucket_Poor,AQI_Bucket_Satisfactory,AQI_Bucket_Very Poor
0,125095.5,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,...,False,False,False,False,True,False,False,False,False,False
1,125095.5,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,...,False,False,False,False,True,False,False,False,False,False
2,125095.5,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,...,False,False,False,True,False,False,False,False,False,False
3,125095.5,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,...,False,False,False,False,True,False,False,False,False,False
4,125095.5,3.0,2747.0,258.36,0.09,2007.0,15.0,16.0,18.0,6.0,...,False,False,False,True,False,False,False,False,False,False


# 12. Save the Feature Engineered Data
Save the processed DataFrame to a CSV file for further modeling and analysis.

In [12]:
output_path = os.path.join("feature_engineered.csv")
df.to_csv(output_path, index=False)
print(f"Feature engineered data saved to {output_path}")

Feature engineered data saved to feature_engineered.csv


# Improved Encoding and Location Filtering
We will encode categorical columns using label encoding based on the actual data values, and filter the dataset to keep only rows where location contains 'Bangalore' (case-insensitive). The 'id' column will be removed, but 'location' will be retained for filtering.

In [ ]:
# Remove 'id' column if present
df = df.drop(columns=[col for col in ['id', 'ID'] if col in df.columns])

# Keep only rows where location contains 'bangalore' (case-insensitive)
if 'location' in df.columns:
    df = df[df['location'].str.lower().str.contains('bangalore')]

# Label encode categorical columns based on actual data
from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
df.head()